# SDS 03 — Distinct Counting & Count-Min Sketch Reference

**Purpose:** reusable exam/reference notebook for Flajolet-Martin intuition, HyperLogLog, and Count-Min Sketch.

This notebook has two layers:
1. small from-scratch implementations for understanding and sanity checks;
2. Spark patterns that keep large data distributed.

No third-party hashing package such as `mmh3` is required.


## 0. Recognition

- **How many distinct values?** → FM / HyperLogLog
- **How many times did key x occur?** → Count-Min Sketch
- **Exactly k uniformly sampled items?** → Reservoir sampling
- **Membership, no false negatives?** → Bloom filter


In [ ]:
import math, hashlib, random
from dataclasses import dataclass
from typing import Iterable, List


## 1. Stable 64-bit hash for local sanity tests

For Spark data, prefer Spark's `xxhash64`. For local reference code we can use Python's standard-library BLAKE2b to obtain a deterministic 64-bit value.


In [ ]:
def stable_hash64(x, salt=''):
    data = (salt + '|' + str(x)).encode('utf-8')
    return int.from_bytes(hashlib.blake2b(data, digest_size=8).digest(), 'big', signed=False)

def leading_zeros_64(x: int) -> int:
    if x == 0:
        return 64
    return 64 - x.bit_length()

for s in ['A','B','C']:
    h = stable_hash64(s)
    print(s, hex(h), leading_zeros_64(h))


## 2. Flajolet-Martin intuition

A simple FM-style statistic tracks the largest number of leading zeros observed. The crude scale is about `2**R`. This is intentionally noisy; HLL improves the idea.


In [ ]:
def fm_crude_estimate(items: Iterable[str]) -> int:
    R = 0
    for x in set(items):  # set only for this tiny local demonstration
        R = max(R, leading_zeros_64(stable_hash64(x)))
    return 2 ** R

items = [f'id-{i}' for i in range(1000)]
print('crude FM estimate:', fm_crude_estimate(items))


## 3. Minimal HyperLogLog from scratch

For each 64-bit hash:
- first `p` bits choose one of `m=2**p` registers;
- remaining bits determine `rho = leading_zeros + 1`;
- register stores the maximum rho seen.

Estimator:

`E = alpha_m * m**2 / sum(2**(-M[j]))`

Small-range correction when many registers are zero:

`E = m * log(m/V)` where `V` is the number of zero registers.


In [ ]:
class HyperLogLog:
    def __init__(self, p=10):
        if not (4 <= p <= 20):
            raise ValueError('Use a reasonable p, e.g. 10-14 for study examples')
        self.p = p
        self.m = 1 << p
        self.M = [0] * self.m

    @staticmethod
    def _alpha(m):
        if m == 16: return 0.673
        if m == 32: return 0.697
        if m == 64: return 0.709
        return 0.7213 / (1.0 + 1.079 / m)

    def add(self, x):
        h = stable_hash64(x)
        j = h >> (64 - self.p)                # first p bits
        remainder_bits = 64 - self.p
        w = h & ((1 << remainder_bits) - 1)  # remaining bits
        rho = remainder_bits + 1 if w == 0 else (remainder_bits - w.bit_length() + 1)
        if rho > self.M[j]:
            self.M[j] = rho

    def estimate(self):
        m = self.m
        raw = self._alpha(m) * m * m / sum(2.0 ** (-v) for v in self.M)
        V = self.M.count(0)
        # classical small-range correction
        if raw <= 2.5 * m and V > 0:
            return m * math.log(m / V)
        return raw

    def merge(self, other):
        if self.p != other.p:
            raise ValueError('HLL sketches must have the same p')
        out = HyperLogLog(self.p)
        out.M = [max(a,b) for a,b in zip(self.M, other.M)]
        return out

hll = HyperLogLog(p=10)
for i in range(10000):
    hll.add(f'user-{i}')
print('m=', hll.m, 'approx RSE=', 1.04/math.sqrt(hll.m))
print('estimate=', round(hll.estimate()), 'true=', 10000)


## 4. HLL parameter helper


In [ ]:
def hll_params_for_rse(target_rse: float):
    needed = (1.04 / target_rse) ** 2
    p = math.ceil(math.log2(needed))
    m = 1 << p
    achieved = 1.04 / math.sqrt(m)
    return {'p': p, 'm': m, 'approx_rse': achieved}

for target in [0.05, 0.03, 0.02, 0.01]:
    print(target, hll_params_for_rse(target))


## 5. Merge sanity check

HLL is mergeable by register-wise maximum.


In [ ]:
a = HyperLogLog(p=10)
b = HyperLogLog(p=10)
for i in range(0,5000): a.add(f'x-{i}')
for i in range(5000,10000): b.add(f'x-{i}')
merged = a.merge(b)
print('merged estimate=', round(merged.estimate()))


## 6. Count-Min Sketch parameters

Classic parameterization:

- `w = ceil(e/epsilon)`
- `d = ceil(log(1/delta))`

Guarantee for insertion-only streams:

`f(x) <= f_hat(x) <= f(x) + epsilon*N` with probability at least `1-delta`.


In [ ]:
def cms_params(epsilon: float, delta: float):
    return math.ceil(math.e / epsilon), math.ceil(math.log(1.0 / delta))

print(cms_params(0.001, 0.01))  # expected (2719, 5)


## 7. Minimal Count-Min Sketch from scratch


In [ ]:
class CountMinSketch:
    def __init__(self, epsilon=0.001, delta=0.01):
        self.epsilon = epsilon
        self.delta = delta
        self.w, self.d = cms_params(epsilon, delta)
        self.C = [[0]*self.w for _ in range(self.d)]
        self.N = 0

    def _idx(self, x, row):
        return stable_hash64(x, salt=f'cms-row-{row}') % self.w

    def add(self, x, count=1):
        if count < 0:
            raise ValueError('This reference implementation assumes insertion-only nonnegative counts')
        for row in range(self.d):
            self.C[row][self._idx(x,row)] += count
        self.N += count

    def query(self, x):
        return min(self.C[row][self._idx(x,row)] for row in range(self.d))

    def merge(self, other):
        if (self.w,self.d,self.epsilon,self.delta) != (other.w,other.d,other.epsilon,other.delta):
            raise ValueError('CMS sketches must have identical dimensions/configuration')
        out = CountMinSketch(self.epsilon, self.delta)
        out.C = [[a+b for a,b in zip(ra,rb)] for ra,rb in zip(self.C,other.C)]
        out.N = self.N + other.N
        return out

cms = CountMinSketch(epsilon=0.01, delta=0.01)
for x in ['A','B','A','C','A','B']:
    cms.add(x)
for x in ['A','B','C','D']:
    print(x, cms.query(x))
print('w,d=',cms.w,cms.d,'N=',cms.N)


## 8. Why the minimum?

In an insertion-only stream, collisions only add extra counts. Every row's counter for x is therefore at least the true count. The minimum picks the least contaminated row.


## 9. Spark: exact vs approximate distinct count

Use these when the question asks for a result. If it asks you to explain/implement HLL, still explain the algorithm.


In [ ]:
from pyspark.sql import functions as F

# Exact
df.agg(F.countDistinct('class_code').alias('exact_distinct'))

# Approximate; rsd is relative standard deviation
df.agg(F.approx_count_distinct('class_code', rsd=0.03).alias('approx_distinct'))


## 10. Spark-native Count-Min Sketch build pattern

This produces a sparse DataFrame of nonzero sketch cells for a batch or micro-batch. The large event table stays distributed.


In [ ]:
from pyspark.sql import functions as F

def build_cms_df(df, item_col, epsilon=0.001, delta=0.01):
    W = math.ceil(math.e / epsilon)
    D = math.ceil(math.log(1.0 / delta))
    rows = spark.range(D).withColumnRenamed('id', 'depth')
    positions = (
        df.select(F.col(item_col).cast('string').alias('item'))
          .crossJoin(rows)
          .withColumn('col_idx', F.pmod(F.xxhash64('item', 'depth'), F.lit(W)))
    )
    sketch = (
        positions.groupBy('depth','col_idx')
                 .agg(F.count(F.lit(1)).alias('count'))
    )
    return sketch, W, D

# sketch, W, D = build_cms_df(events, 'class_code')


## 11. Query a Spark CMS sketch

The query key is tiny, so computing its D positions locally is fine if we use the same deterministic hash definition. To avoid mismatches, the simplest exam-safe approach is to let Spark compute those positions too.


In [ ]:
def cms_query_df(sketch_df, key, W, D):
    q = spark.createDataFrame([(str(key),)], ['item'])
    rows = spark.range(D).withColumnRenamed('id','depth')
    qpos = (
        q.crossJoin(rows)
         .withColumn('col_idx', F.pmod(F.xxhash64('item','depth'), F.lit(W)))
         .select('depth','col_idx')
    )
    vals = (
        qpos.join(sketch_df, ['depth','col_idx'], 'left')
            .fillna({'count':0})
            .agg(F.min('count').alias('estimate'))
            .first()['estimate']
    )
    return int(vals)


## 12. Merge Spark CMS partial sketches

Partial CMS tables merge by summing matching cells.


In [ ]:
def merge_cms_dfs(a, b):
    return (
        a.unionByName(b)
         .groupBy('depth','col_idx')
         .agg(F.sum('count').alias('count'))
    )


## 13. Spark HLL register construction — explicit teaching version

Spark already has `approx_count_distinct`, but if you need to demonstrate the register idea, this pattern uses Spark `xxhash64` and a small Python UDF to split the signed 64-bit hash into an unsigned register index and rho. The UDF executes on workers; it does not collect the stream to the driver.


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType

P = 10
M = 1 << P
R = 64 - P

parts_schema = StructType([
    StructField('register', IntegerType(), False),
    StructField('rho', IntegerType(), False),
])

@F.udf(parts_schema)
def hll_parts_from_signed_long(h):
    u = int(h) & ((1 << 64) - 1)
    j = u >> (64 - P)
    w = u & ((1 << R) - 1)
    rho = R + 1 if w == 0 else (R - w.bit_length() + 1)
    return (int(j), int(rho))

registers = (
    df.select(F.xxhash64(F.col('class_code').cast('string')).alias('h'))
      .withColumn('parts', hll_parts_from_signed_long('h'))
      .select(F.col('parts.register').alias('register'), F.col('parts.rho').alias('rho'))
      .groupBy('register')
      .agg(F.max('rho').alias('M'))
)


## 14. Exam critique template — HLL

**Correct:** appropriate distinct-counting family; leading-zero intuition; multiple registers reduce variance.

**Limitations to check:** missing `p/m`; missing hash width; wrong estimator; missing small-range correction; no accuracy statement; non-Spark/external dependency.


## 15. Exam critique template — CMS

Check:
- Is the problem really frequency estimation?
- Are `epsilon`, `delta`, `w`, and `d` explicit?
- Is update one counter per row?
- Is query the **minimum**, not average/maximum?
- Is the guarantee additive in total stream mass `N`?
- Is the implementation insertion-only or does it claim unsupported deletion behavior?


## 16. Fast lookup formulas

- HLL: `m=2**p`
- HLL RSE: `1.04/sqrt(m)`
- HLL raw estimate: `alpha_m*m**2 / sum(2**(-M_j))`
- HLL small-range: `m*log(m/V)`
- CMS width: `ceil(e/epsilon)`
- CMS depth: `ceil(log(1/delta))`
- CMS query: minimum across rows
- CMS error: at most `epsilon*N` additive error with probability at least `1-delta`
